In [15]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import welch

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 1. FEATURE EXTRACTION
# =========================
def extract_features(signal):
    signal = np.array(signal)

    mean = np.mean(signal)
    std = np.std(signal)
    sk = skew(signal)
    kurt = kurtosis(signal)

    freqs, psd = welch(signal)
    psd_norm = psd / np.sum(psd)
    entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))

    rms = np.sqrt(np.mean(signal**2))
    max_val = np.max(signal)
    min_val = np.min(signal)
    p2p = max_val - min_val

    crest = max_val / (rms + 1e-12)
    clearance = max_val / (np.mean(np.sqrt(np.abs(signal)))**2 + 1e-12)
    shape = rms / (np.mean(np.abs(signal)) + 1e-12)
    impulse = max_val / (np.mean(np.abs(signal)) + 1e-12)

    return [mean, std, sk, kurt, entropy, rms, max_val, p2p,
            crest, clearance, shape, impulse]


# =========================
# 2. LOAD DATA
# =========================
df = pd.read_csv("/content/RUL_Estimationedited - RUL_Estimation.csv.csv")

X = df.drop(columns=["RUL(min)"])
y = df["RUL(min)"]


# =========================
# 3. TIME-SAFE SPLIT
# =========================
split = int(0.8 * len(df))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


# =========================
# 4. RF → FEATURE IMPORTANCE
# =========================
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

importances = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print("\nFeature Importance:\n", feature_importance_df)


# =========================
# 5. SELECT TOP FEATURES
# =========================
top_k = 6
top_features = feature_importance_df["feature"].head(top_k).tolist()

print("\nSelected Features:", top_features)

X_train = X_train[top_features]
X_test = X_test[top_features]


# =========================
# 6. NORMALIZATION
# =========================
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

y_mean = y_train.mean()
y_std = y_train.std()

y_train = (y_train - y_mean) / y_std
y_test = (y_test - y_mean) / y_std


# =========================
# 7. SVM MODEL
# =========================
model = SVR(
    kernel='rbf',
    C=50,
    epsilon=0.05,
    gamma='scale'
)

model.fit(X_train, y_train)


# =========================
# 8. EVALUATION
# =========================
y_pred = model.predict(X_test)

# Denormalize
y_pred = y_pred * y_std + y_mean
y_test = y_test * y_std + y_mean

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
accuracy = np.mean(1 - np.abs((y_test - y_pred) / (y_test + 1e-6))) * 100

print("\nFinal Results (SVM):")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Accuracy (%):", accuracy)


# =========================
# 9. PREDICTION FUNCTION
# =========================
def predict_rul_from_signal(signal):
    features = extract_features(signal)

    # map feature names → values
    feature_names = df.drop(columns=["RUL(min)"]).columns.tolist()
    feature_dict = dict(zip(feature_names, features))

    selected = [feature_dict[f] for f in top_features]

    selected = scaler_X.transform([selected])

    pred = model.predict(selected)[0]

    return pred * y_std + y_mean



Feature Importance:
        feature  importance
0   Unnamed: 0    0.994934
6          rms    0.002438
2          std    0.001690
1         mean    0.000703
10   clearence    0.000088
11       shape    0.000048
5      entropy    0.000040
4     kurtosis    0.000029
8          p2p    0.000009
3         skew    0.000007
7          max    0.000006
12     impulse    0.000004
9        crest    0.000004

Selected Features: ['Unnamed: 0', 'rms', 'std', 'mean', 'clearence', 'shape']

Final Results (SVM):
MAE: 2236.2319327157747
RMSE: 2696.117159186984
R2: -21.476956512252602
Accuracy (%): -2278715894.288447


In [ ]:


# =========================
# 10. EXAMPLE
# =========================
raw_signal = np.random.randn(1024)

rul = predict_rul_from_signal(raw_signal)
print("\nPredicted RUL:", rul)